# DR100 quick start

This notebook walks through common uses of the DR100 resource table for
building drug-focused AI agents.

1. Load the three resource files (Drug-centric, Drug-related, Auxiliary)
2. Filter by knowledge interface (`pathway` field), task tag, identifier, or access approach
3. Build example resource short-lists
4. Cross-tabulate knowledge interface by subcategory

**Requirements**: `pip install pandas jsonschema`

## 1. Load the resource table

In [ ]:
import json
from pathlib import Path
import pandas as pd

REGISTRY = Path('..') / 'registry'

drug_centric = pd.DataFrame(json.loads((REGISTRY / 'drug_centric.json').read_text()))
drug_related = pd.DataFrame(json.loads((REGISTRY / 'drug_related.json').read_text()))
auxiliary    = pd.DataFrame(json.loads((REGISTRY / 'auxiliary.json').read_text()))

print(f'drug-centric:  {len(drug_centric):>4d} records')
print(f'drug-related:  {len(drug_related):>4d} records')
print(f'auxiliary:     {len(auxiliary):>4d} records')
print(f'TOTAL:         {len(drug_centric) + len(drug_related) + len(auxiliary):>4d} records')

drug_centric.head()

## 2. Knowledge interfaces (`pathway` field)

In [ ]:
pathway_order = ['Direct Prompt', 'RAG', 'Graph Query', 'REST', 'Computational', 'Hybrid']
drug_centric['pathway'].value_counts().reindex(pathway_order)

## 3. Example short-list: pharmacogenomic dosing

Relevant task tags: T1 (identification), T2 (labeling), T3 (mechanism), T6 (target binding).
Filter resources by relevant subcategory, open access, and a recent update status.

In [ ]:
pgx_subcats = ['Pharmacogenomics', 'Drug Knowledgebase', 'Drug Ontology/Terminology', 'Drug Labeling/Info']
pgx_shortlist = drug_centric[
    drug_centric['subcategory'].isin(pgx_subcats)
    & drug_centric['access'].isin(['Open API', 'Open Download', 'Academic API'])
    & drug_centric['update_cadence'].isin(['Continuous', 'Periodic Recent'])
][['name', 'subcategory', 'pathway', 'identifier', 'access', 'update_cadence', 'link']]
pgx_shortlist

## 4. Example short-list: drug–drug interaction checking

Relevant task tags: T5 (drug–drug interaction), T1 (identification), T4 (safety).

In [ ]:
ddi_subcats = ['Drug-Drug Interaction (DDI)', 'Drug Knowledgebase', 'Adverse Drug Reaction (ADR)']
ddi_shortlist = drug_centric[
    drug_centric['subcategory'].isin(ddi_subcats)
    & drug_centric['access'].isin(['Open API', 'Open Download'])
][['name', 'subcategory', 'pathway', 'access', 'update_cadence']]
ddi_shortlist

## 5. Cross-tab: knowledge interface (`pathway` field) × subcategory

In [ ]:
ct = pd.crosstab(drug_centric['subcategory'], drug_centric['pathway']).reindex(columns=pathway_order, fill_value=0)
ct['TOTAL'] = ct.sum(axis=1)
ct.sort_values('TOTAL', ascending=False)

## 6. Schema validation (round-trip check)

In [ ]:
from jsonschema import Draft7Validator

schema = json.loads((Path('..') / 'schema' / 'dr100_schema.json').read_text())
v = Draft7Validator(schema)

for fn in ['drug_centric', 'drug_related', 'auxiliary']:
    data = json.loads((REGISTRY / f'{fn}.json').read_text())
    errs = [e for rec in data for e in v.iter_errors(rec)]
    print(f'{fn:>15s}: {len(data):>3d} records, {len(errs)} validation errors')

## Next steps

- Browse the construction log: `docs/data_quality_report.md`
- Propose a new resource: see `docs/contribution_guide.md`
- Read the Review's section on knowledge interfaces (Figure 4) for their definitions